# Build the offline knowledge base on Colab (free)
Produces `knowledge.sqlite` and saves it to your Google Drive; download it on your phone and use **Import knowledge base** in the app.

**Tip:** start with `20231101.simple` (small, quick). Later switch to `20231101.en` with `MAX_ARTICLES` to fit your phone's storage and your data plan.
Colab sessions are temporary and can disconnect; if the full build is too slow, lower `MAX_ARTICLES`.

In [ ]:
GITHUB_USER = 'your-username'   # <- edit
REPO = 'offline-research'
CONFIG = '20231101.simple'      # or '20231101.en'
MAX_ARTICLES = 0                # 0 = no limit; e.g. 300000 for a partial English index
MIN_CHARS = 800                 # use 2500 for the English set
NAME = 'Simple Wikipedia'

In [ ]:
!pip -q install pyarrow huggingface_hub
!git clone -q https://github.com/{GITHUB_USER}/{REPO} /content/repo

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id='wikimedia/wikipedia', repo_type='dataset',
                  allow_patterns=[f'{CONFIG}/*'], local_dir='/content/wiki')
!ls -la /content/wiki/{CONFIG} | head

In [ ]:
!python /content/repo/scripts/build_index.py --source parquet --input "/content/wiki/{CONFIG}/*.parquet" \
    --out /content/knowledge.sqlite --name "{NAME}" --min-chars {MIN_CHARS} --max-chunks 6 --skip-lists --max-articles {MAX_ARTICLES}
!ls -lh /content/knowledge.sqlite

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/knowledge.sqlite /content/drive/MyDrive/knowledge.sqlite
print('Saved to Google Drive: My Drive/knowledge.sqlite')

On your phone: open the Google Drive app -> `knowledge.sqlite` -> **Download**. Then in Offline Research tap **Import knowledge base** and pick it from Downloads. Delete the Downloads copy afterwards to free space.